# Post-training experiments on SmolLM-135M

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s1mran/finetuning/blob/main/run_in_colab.ipynb)

Three families of run, each answering one question:

| folder | question | scripts |
|---|---|---|
| `CPT/` | does CPT-first adaptation work? | `cpt_fin_then_sft.py`, `cpt_med_then_sft.py` |
| `SFT/` | what does reversing the order cost? | `sft_fin_then_cpt.py`, `sft_med_then_cpt.py` |
| `RLHF/` | can DPO align it without reward hacking? | `sft_then_dpo.py` |

Artifacts land in `<TYPE>/reports/<run>/`, resolved from each script's own
location — so it does not matter which directory you launch from.

**Before you run anything:** Runtime → Change runtime type → **T4 GPU**.
Unsloth is CUDA-only; there is no CPU or Apple-MPS fallback.

Cells 1–3 are setup and must be re-run after any restart or VM recycle — Colab
wipes installed packages every time.

## 1. Confirm the GPU is actually attached

This must print a GPU table. If it errors you are on a CPU runtime and
everything below fails at `import unsloth` — fix the runtime type first.

In [ ]:
!nvidia-smi

## 2. Install dependencies

~2 minutes, and needed again after every restart. No kernel restart required
afterwards: the training scripts run as `!python` subprocesses, which pick up
the newly installed packages.

In [ ]:
!pip install -q unsloth trl peft transformers datasets accelerate bitsandbytes

## 3. Get the code

Clones on a fresh VM, pulls if it is already there.

In [ ]:
!git clone https://github.com/s1mran/finetuning.git /content/finetuning 2>/dev/null || git -C /content/finetuning pull
!find /content/finetuning -name '*.py' | sort

## 4. Smoke test — 30 steps per stage

~2 minutes. Checks the whole pipeline before you spend 15 on it.

**Four things to verify in the output:**

1. `[cpt-data] general replay: Salesforce/wikitext, N chunks` — if this says
   *unavailable* the script now stops rather than silently reporting a fake
   perplexity.
2. `[ppl] base -> {'domain': ..., 'general': <a real number>}` — a general
   perplexity of exactly `1.0` means an empty corpus, not a perfect model.
3. `[cpt-data / domain]` and `[sft-data / ...]` blocks printing **real training
   strings**. Read them. A renamed column or an empty context is invisible in a
   loss curve and obvious here.
4. `[mask] supervising N/M tokens` — response-only masking is live, and should
   be a *minority* of tokens.

In [ ]:
!python /content/finetuning/CPT/src/cpt_fin_then_sft.py --smoke

## 5. The ordering experiment — medical

Run these two as a matched pair: same seed, same step counts, only the stage
order differs. ~15 minutes each.

The medical pair is the more informative comparison — its CPT corpus
(`epfl-llm/guidelines`) is genuinely distinct from its SFT corpus. See the
closing cell for why the finance pair currently is not.

In [ ]:
!python /content/finetuning/CPT/src/cpt_med_then_sft.py

In [ ]:
!python /content/finetuning/SFT/src/sft_med_then_cpt.py

## 6. The ordering experiment — finance (optional)

The original pair. Keep step counts matched across the two.

In [ ]:
!python /content/finetuning/CPT/src/cpt_fin_then_sft.py

In [ ]:
!python /content/finetuning/SFT/src/sft_fin_then_cpt.py

## 7. Compare a matched pair

Reads both `report.json` files and lays the numbers side by side. Perplexity
columns should be comparable; probe behaviour should not be. Set `DOMAIN` to
whichever pair you ran.

In [ ]:
import json, math
from pathlib import Path

ROOT = Path("/content/finetuning")
DOMAIN = "med"          # "med" or "fin"

# (report path, stage-1 ppl key, stage-2 ppl key, stage-1 probe key)
LAYOUT = {
    "CPT -> SFT": (ROOT / "CPT/reports" / f"cpt_{DOMAIN}_then_sft" / "report.json",
                   "ppl_after_cpt", "ppl_after_sft", "probes_after_cpt"),
    "SFT -> CPT": (ROOT / "SFT/reports" / f"sft_{DOMAIN}_then_cpt" / "report.json",
                   "ppl_after_sft", "ppl_after_cpt", "probes_after_sft"),
}

def num(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "  n/a "
    return f"{v:6.2f}"

reports = {}
for name, (path, *_rest) in LAYOUT.items():
    if path.exists():
        reports[name] = json.loads(path.read_text())
    else:
        print(f"missing {path} -- run that cell first")

for name, rep in reports.items():
    _, s1, s2, s1_probe = LAYOUT[name]
    print("=" * 66)
    print(f"{name}   (stage 1 -> stage 2)")
    print("=" * 66)
    for dom in ("domain", "general"):
        b = rep.get("ppl_before", {}).get(dom)
        a = rep.get(s1, {}).get(dom)
        c = rep.get(s2, {}).get(dom)
        print(f"  {dom:>7} ppl : {num(b)}  ->{num(a)}  ->{num(c)}")
    for stage, key in (("base", "probes_base"), ("stage1", s1_probe),
                       ("final", "probes_final")):
        rate = (rep.get(key) or {}).get("eos_rate")
        if rate is not None:
            print(f"  {stage:>7} eos : {rate:.0%}")
    print()

# The qualitative half: does the final model answer *and stop*?
if reports:
    print("=" * 66)
    print("FINAL ALPACA PROBES")
    print("=" * 66)
    keys = {k for r in reports.values()
            for k in (r.get("probes_final") or {}) if k.startswith("alpaca::")}
    for pk in sorted(keys):
        print(f"\n  {pk.split('::', 1)[1]}")
        for name, rep in reports.items():
            print(f"    [{name}] {(rep.get('probes_final') or {}).get(pk, '<missing>')[:240]}")

## 8. Preference alignment — SFT → DPO

~20 minutes. SFT on empathetic dialogue, then DPO on human preference pairs.

The interesting output is the **BEFORE / AFTER** table at the end. Read the
length and repetition columns *before* the accuracy column:

- preference accuracy rising, length and repetition flat → alignment worked
- preference accuracy rising, mean tokens climbing sharply → length exploitation
- preference accuracy rising, repetition climbing → degeneration
- general perplexity climbing sharply → drifted off the reference

The script prints an explicit warning if mean response length grew more than
1.5×. If it fires, raise `--beta` (tighter KL leash) or lower
`--max-length-ratio` (stricter length filter on the preference pairs) and
re-run.

In [ ]:
!python /content/finetuning/RLHF/src/sft_then_dpo.py --smoke

In [ ]:
!python /content/finetuning/RLHF/src/sft_then_dpo.py

Tightening the guards, if the warning fired:

In [ ]:
!python /content/finetuning/RLHF/src/sft_then_dpo.py --beta 0.3 --max-length-ratio 1.2

## 9. Keep the results

Colab recycles VMs on idle and caps session length, so anything under
`/content` is temporary.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DEST = "/content/drive/MyDrive/finetuning_reports"
!mkdir -p "$DEST"
!cp -r /content/finetuning/CPT/reports "$DEST/CPT"
!cp -r /content/finetuning/SFT/reports "$DEST/SFT"
!cp -r /content/finetuning/RLHF/reports "$DEST/RLHF"
!ls -R "$DEST" | head -40

## 10. Push a model to the Hugging Face Hub

Needs a **write**-scoped token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens),
stored in Colab's secrets pane (🔑 in the left sidebar) as `HF_TOKEN`.

To publish a model you **already trained**, without retraining — point
`FOLDER` at any `04_final_merged/` (or an adapter directory):

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

FOLDER = "/content/finetuning/CPT/reports/cpt_med_then_sft/04_final_merged"
REPO = "sidhusarkar/smollm-135m-med-cpt-then-sft"      # your username/repo

token = userdata.get("HF_TOKEN")
api = HfApi()
api.create_repo(REPO, repo_type="model", exist_ok=True, token=token)
api.upload_folder(folder_path=FOLDER, repo_id=REPO, repo_type="model", token=token)
print(f"https://huggingface.co/{REPO}")

Or push as part of a fresh run, which also generates a model card:

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!python /content/finetuning/CPT/src/cpt_med_then_sft.py --push-to-hub

## Known limitation

`eloukas/edgar-corpus` — the raw 10-K prose the **finance** CPT stage wants — is
script-based, and `datasets` dropped support for script loaders. Every EDGAR
mirror checked has the same problem, so the finance runs fall back to the
`context` column of `virattt/financial-qa-10K`, *the same corpus their SFT stage
trains on*.

That matches what was actually observed: domain perplexity 22.0 → 19.9 → 20.3,
a weak drop that partly reverses, rather than the sharp drop the design
predicts.

The **medical** runs do not have this problem — `epfl-llm/guidelines` is
genuinely distinct from the ChatDoctor SFT data — so treat the medical pair as
the informative ordering comparison until a loadable EDGAR mirror turns up.